# 01 — EDA y clustering piloto (1.000 usuarios)

**Proyecto de Grado EAFIT** — 5 categorías de riesgo de retiro (KKBox)

Abrir en **Google Colab**: Archivo → Subir notebook, o abrir desde GitHub.

## Objetivos de esta sesión
1. Inventariar variables en `transactions` y `members`
2. Construir features numéricas por `msno`
3. K-Means con k=5 sobre muestra piloto
4. Caracterizar % `is_churn` por cluster y exportar 5 hojas Excel

## 0. Setup Colab

In [ ]:
# Descomentar en Colab si el repo no está montado:
# !git clone https://github.com/danielrpo1/pdgrado.git
# %cd pdgrado

!pip install -q kaggle openpyxl scikit-learn seaborn

import sys
from pathlib import Path

ROOT = Path('.').resolve()
if (ROOT / 'src').exists():
    sys.path.insert(0, str(ROOT))
else:
    ROOT = Path('/content/pdgrado')  # tras git clone
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import PILOT_SAMPLE_SIZE, RANDOM_STATE, N_CLUSTERS
from src.features import build_user_features
from src.clustering import (
    fit_kmeans,
    churn_rate_by_cluster,
    order_clusters_by_risk,
    export_clusters_to_excel,
)

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 30)

## 1. Descargar datos (Kaggle API)

1. Crear API token en https://www.kaggle.com/settings → **Create New Token**
2. En Colab: subir `kaggle.json` o usar secrets

Competencia: `kkbox-churn-prediction-challenge`

In [ ]:
import os
from pathlib import Path

DATA_DIR = ROOT / 'data' / 'raw'
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Opción A: Colab — subir kaggle.json manualmente
# from google.colab import files
# uploaded = files.upload()
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Opción B: si ya descargaste CSV localmente, copia a data/raw/

USE_KAGGLE_DOWNLOAD = False  # cambiar a True en Colab con credenciales

if USE_KAGGLE_DOWNLOAD:
    os.environ['KAGGLE_CONFIG_DIR'] = str(Path.home() / '.kaggle')
    !kaggle competitions download -c kkbox-churn-prediction-challenge -p {DATA_DIR}
    # Descomprimir .zip según archivos descargados

required = ['train.csv', 'transactions.csv', 'members.csv']
missing = [f for f in required if not (DATA_DIR / f).exists()]
if missing:
    print('Faltan archivos en data/raw/:', missing)
    print('Descarga manual: https://www.kaggle.com/competitions/kkbox-churn-prediction-challenge/data')
else:
    print('Datos listos en', DATA_DIR)

## 2. Carga y muestra piloto (1.000 usuarios)

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
members = pd.read_csv(DATA_DIR / 'members.csv')
transactions = pd.read_csv(DATA_DIR / 'transactions.csv')

print('train:', train.shape)
print('members:', members.shape)
print('transactions:', transactions.shape)
print('\nColumnas transactions:\n', transactions.columns.tolist())
print('\nColumnas members:\n', members.columns.tolist())

In [ ]:
# Muestra estratificada por is_churn para mantener proporción de churn
pilot_msno = (
    train.groupby('is_churn', group_keys=False)
    .apply(lambda x: x.sample(
        min(len(x), PILOT_SAMPLE_SIZE // 2),
        random_state=RANDOM_STATE
    ))
    ['msno']
    .drop_duplicates()
)
if len(pilot_msno) > PILOT_SAMPLE_SIZE:
    pilot_msno = pilot_msno.sample(PILOT_SAMPLE_SIZE, random_state=RANDOM_STATE)

tx_pilot = transactions[transactions['msno'].isin(pilot_msno)]
print(f'Usuarios piloto: {pilot_msno.nunique()} | Filas transactions: {len(tx_pilot)}')

## 3. Feature engineering (numéricas)

In [ ]:
features = build_user_features(tx_pilot)
df = features.merge(train[['msno', 'is_churn']], on='msno', how='left')
df = df.merge(members, on='msno', how='left')

feature_cols = [
    c for c in df.columns
    if c not in ('msno', 'is_churn', 'city', 'bd', 'gender', 'registered_via', 'registration_init_time')
    and df[c].dtype in ('int64', 'float64', 'int32', 'float32')
]
X = df[feature_cols].fillna(0)
print('Features para clustering:', feature_cols)
df[['msno', 'is_churn'] + feature_cols[:8]].head()

## 4. K-Means (k=5)

In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

model, scaler, labels = fit_kmeans(X)
df['cluster_raw'] = labels.values

X_scaled = scaler.transform(X)
sil = silhouette_score(X_scaled, labels)
print(f'Silhouette (k={N_CLUSTERS}): {sil:.3f}')

risk_map = order_clusters_by_risk(df, cluster_col='cluster_raw')
df['risk_category'] = df['cluster_raw'].map(risk_map)

summary = churn_rate_by_cluster(df, cluster_col='risk_category')
print('\nTasa de churn por categoría de riesgo (0=bajo → 4=alto):')
display(summary)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
summary['churn_rate'].plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('% is_churn por categoría de riesgo')
ax.set_xlabel('Categoría de riesgo')
ax.set_ylabel('Tasa churn')
plt.tight_layout()
plt.show()

## 5. Caracterización (variables categóricas)

In [ ]:
for cat in sorted(df['risk_category'].unique()):
    sub = df[df['risk_category'] == cat]
    print(f'\n=== Riesgo {int(cat)} | n={len(sub)} | churn={sub["is_churn"].mean():.2%} ===')
    if 'city' in sub.columns:
        print('Top ciudades:', sub['city'].value_counts().head(3).to_dict())
    if 'gender' in sub.columns:
        print('Género:', sub['gender'].value_counts().to_dict())
    print('Medias numéricas clave:')
    print(sub[feature_cols[:6]].mean().round(2).to_string())

## 6. Exportar 5 hojas Excel

In [ ]:
out_dir = ROOT / 'outputs' / 'clusters'
excel_path = export_clusters_to_excel(df, out_dir)
print('Exportado:', excel_path)

# Colab: descargar archivo
# from google.colab import files
# files.download(str(excel_path))

## Próximos pasos
- Revisar con asesor perfiles y orden de riesgo
- Completar vacíos en literatura (clasificación multidimensional)
- Fase 2: red neuronal 5 clases con probabilidades